In [1]:
from pathlib import Path
import xml.etree.ElementTree as ET
import pandas as pd
import yaml
from collections import Counter

In [5]:
KAGGLE_ROOT = Path("data/kaggle_raw/src/dataset/dataset_augmented/")
MAPPING_FILE = Path("data/class_mapping.yaml")

xml_files = list(KAGGLE_ROOT.rglob("*.xml"))
print("Total XML files found:", len(xml_files))

Total XML files found: 0


In [ ]:
all_classes = []

for xml_file in xml_files:
    tree = ET.parse(xml_file)
    root = tree.getroot()
    for obj in root.findall("object"):
        all_classes.append(obj.findtext("name"))

unique_classes = sorted(set(all_classes))
print("Total unique original classes:", len(unique_classes))
unique_classes[:20]

In [ ]:
with open(MAPPING_FILE, "r") as f:
    mapping = yaml.safe_load(f)

def map_class(original_name):
    original_name = original_name.lower()
    for target_class, keywords in mapping.items():
        for kw in keywords:
            if kw in original_name:
                return target_class
    return None

mapped = [map_class(c) for c in unique_classes]
mapping_df = pd.DataFrame({
    "original_class": unique_classes,
    "mapped_class": mapped
})

mapping_df.head(20)

In [ ]:
coverage = mapping_df["mapped_class"].value_counts(dropna=False)
coverage